In [ ]:
%%bash
set -e

cd /content

# Download micromamba into /content/bin/micromamba
curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xvj bin/micromamba

# Create a clean env (isolated from Colab’s global packages)
./bin/micromamba create -y -n cb311 -c conda-forge python=3.11 pip

echo "✅ micromamba ready at /content/bin/micromamba"
echo "✅ env created: cb311"


In [ ]:
%%bash
set -euo pipefail

cd /content
MICROMAMBA="/content/bin/micromamba"

ts() { date +"[%Y-%m-%d %H:%M:%S]"; }

echo "$(ts) Sanity check micromamba:"
ls -lah "$MICROMAMBA"

echo "$(ts) Upgrading pip tooling inside cb311..."
"$MICROMAMBA" run -n cb311 python -m pip install -U pip setuptools wheel --progress-bar on

echo "$(ts) Installing PyTorch 2.5.1 (CUDA 12.1)... (this can take a while)"
"$MICROMAMBA" run -n cb311 pip install \
  --progress-bar on \
  torch==2.5.1+cu121 torchaudio==2.5.1+cu121 torchvision==0.20.1+cu121 \
  --index-url https://download.pytorch.org/whl/cu121

echo "$(ts) Installing Chatterbox package (from GitHub, no-cache, upgrade)..."
"$MICROMAMBA" run -n cb311 pip uninstall -y chatterbox-tts chatterbox || true
"$MICROMAMBA" run -n cb311 pip install \
  --no-cache-dir --upgrade \
  --progress-bar on \
  "chatterbox-tts @ git+https://github.com/devnen/chatterbox-v2.git@master"

echo "$(ts) Installing s3tokenizer + onnx (--no-deps to avoid protobuf conflict)..."
"$MICROMAMBA" run -n cb311 pip install --no-deps s3tokenizer==0.3.0 onnx==1.16.0

echo "$(ts) Force-upgrading protobuf for onnx compatibility..."
"$MICROMAMBA" run -n cb311 pip install --no-deps --force-reinstall "protobuf>=4.25.0"

echo "$(ts) ✅ Installation complete!"

In [ ]:
%%bash
set -e

/content/bin/micromamba run -n cb311 python - <<'PY'
import inspect, torch
import chatterbox.tts_turbo as t

print("✅ torch:", torch.__version__)
print("✅ cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("✅ gpu:", torch.cuda.get_device_name(0))

print("✅ chatterbox.tts_turbo path:", t.__file__)

src = inspect.getsource(t.ChatterboxTurboTTS.from_pretrained)
print("\n--- from_pretrained() (first ~80 lines) ---")
print("\n".join(src.splitlines()[:80]))

# Heuristic check for the common buggy pattern that forces token=True semantics
markers = [" or True", "token=True", "token = True", "use_auth_token=True"]
hits = [m for m in markers if m in src]
print("\nHeuristic auth-forcing markers found:", hits)

if hits:
    raise SystemExit(
        "\n❌ This install still appears to force HF auth.\n"
        "Re-run Cell 2 (it already uses --no-cache-dir --upgrade).\n"
    )

print("\n✅ Looks good: Turbo should download without requiring user tokens.")
PY

In [ ]:
# @title 4. Install Server + Run With Full Live Logs (foreground)
import os, time, subprocess, socket, requests, shutil, yaml
from pathlib import Path
from google.colab import userdata

PORT = 8004

# ==== YOUR FORK ====
REPO_OWNER = "michtai"
REPO_NAME = "chatterbox"
GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")  # Colab secret, needed if the fork is private
REPO_DIR = f"/content/{REPO_NAME}"
CLONE_URL = f"https://{GITHUB_TOKEN}@github.com/{REPO_OWNER}/{REPO_NAME}.git"
# =====================

LOG_STDOUT = "/content/chatterbox_server_stdout.log"

# ==== EDIT THESE IF YOU WANT DIFFERENT DEFAULTS ====
DRIVE_ROOT = Path("/content/drive/MyDrive/chatterbox")   # everything lives under here
VOICE_FILENAME = "delightful_really_soft.wav"                           # your reference voice, stored in DRIVE_ROOT
CHUNK_SIZE = 130  # lower than the 240 default — big chunks were cramming multiple
                  # short dialogue turns into one TTS call, causing static/dropped/
                  # garbled audio at chunk boundaries
GENERATION_DEFAULTS = {
    "temperature": 0.65,
    "exaggeration": 0.4,
    "cfg_weight": 0.5,
    "seed": 1818,
    "speed_factor": 1.0,
}
# =====================================================

def sh(cmd, check=False):
    return subprocess.run(["bash", "-lc", cmd], check=check)

def port_open(host="127.0.0.1", port=PORT, timeout=0.25):
    try:
        with socket.create_connection((host, port), timeout=timeout):
            return True
    except OSError:
        return False

os.chdir("/content")

# === Mount Google Drive so outputs (and your voice sample) persist across sessions ===
print("=== Mounting Google Drive ===")
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

DRIVE_OUTPUTS_DIR = DRIVE_ROOT / "outputs"
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
DRIVE_OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
print(f"✅ Chatterbox Drive folder: {DRIVE_ROOT}")
print(f"✅ Generated audio will be saved to: {DRIVE_OUTPUTS_DIR}")

# === Make sure your reference voice is saved in Drive (one-time upload, then reused forever) ===
voice_path_in_drive = DRIVE_ROOT / VOICE_FILENAME
if not voice_path_in_drive.exists():
    print(f"\n⚠️  No voice file found at {voice_path_in_drive}")
    print("Upload your reference voice .wav now — it will be saved to Drive and reused automatically next time.\n")
    from google.colab import files
    uploaded = files.upload()
    if not uploaded:
        raise SystemExit("No file uploaded. Re-run this cell and upload your voice sample.")
    src_name = next(iter(uploaded.keys()))
    shutil.move(src_name, str(voice_path_in_drive))
    print(f"✅ Saved your voice sample to {voice_path_in_drive}")
else:
    print(f"✅ Using existing voice sample from Drive: {voice_path_in_drive}")

# Fresh clone of your fork (already has the sentence-splitting / chunking /
# output-filename patches baked in, so no runtime patching needed here)
sh(f"rm -rf {REPO_DIR}", check=False)
sh(f"git clone {CLONE_URL} {REPO_DIR}", check=True)
os.chdir(REPO_DIR)

print("=== Quick system checks ===")
sh("nvidia-smi || true", check=False)

print("\n=== Installing server requirements (prefer repo pins if present) ===")
if Path("requirements-nvidia.txt").exists():
    sh("/content/bin/micromamba run -n cb311 pip install -U pip setuptools wheel", check=False)
    sh("/content/bin/micromamba run -n cb311 pip install -r requirements-nvidia.txt", check=False)
else:
    sh(
        "/content/bin/micromamba run -n cb311 pip install -U pip setuptools wheel && "
        "/content/bin/micromamba run -n cb311 pip install "
        "fastapi 'uvicorn[standard]' pyyaml soundfile librosa safetensors "
        "python-multipart requests jinja2 watchdog aiofiles unidecode inflect tqdm "
        "pydub audiotsm praat-parselmouth",
        check=False
    )

# Force-upgrade protobuf after requirements install.
# descript-audiotools (pulled by descript-audio-codec) pins protobuf<3.20
# but onnx needs >=3.20.2. The old protobuf lacks the 'builder' module.
sh("/content/bin/micromamba run -n cb311 pip install --no-deps --force-reinstall 'protobuf>=4.25.0'", check=False)

# Patch chatterbox to make Perth watermarker gracefully optional.
# Perth can fail to initialize on some environments; without this patch
# the server crashes with "'NoneType' object is not callable".
# Same patch that start.py applies via _patch_chatterbox_watermarker().
print("\n=== Applying watermarker patch ===")
SITE_PKG = "/root/.local/share/mamba/envs/cb311/lib/python3.11/site-packages"
CB_DIR = Path(SITE_PKG) / "chatterbox"
SENTINEL = "# [patched: watermarker made optional]"
TARGET = "self.watermarker = perth.PerthImplicitWatermarker()"
patched = 0
for fname in ["tts.py", "tts_turbo.py", "mtl_tts.py", "vc.py"]:
    fp = CB_DIR / fname
    if not fp.exists():
        continue
    content = fp.read_text(encoding="utf-8")
    if SENTINEL in content or TARGET not in content:
        continue
    lines = content.split("\n")
    new_lines = []
    for line in lines:
        if TARGET in line and line.lstrip().startswith("self."):
            ind = line[:len(line) - len(line.lstrip())]
            new_lines.append(f"{ind}{SENTINEL}")
            new_lines.append(f"{ind}try:")
            new_lines.append(f"{ind}    self.watermarker = perth.PerthImplicitWatermarker()")
            new_lines.append(f"{ind}except Exception:")
            new_lines.append(f"{ind}    class _NoOpWatermarker:")
            new_lines.append(f"{ind}        def apply_watermark(self, wav, *args, **kwargs):")
            new_lines.append(f"{ind}            return wav")
            new_lines.append(f"{ind}    self.watermarker = _NoOpWatermarker()")
        else:
            new_lines.append(line)
    fp.write_text("\n".join(new_lines), encoding="utf-8")
    print(f"  Patched {fname}")
    patched += 1
if patched:
    print(f"  {patched} file(s) patched for optional watermarking")
else:
    print("  No patching needed")

# === Replace the built-in sample voices with your own, and point the server at Drive ===
print("\n=== Configuring voice + output folder + generation defaults ===")

VOICES_DIR = Path("voices")
REFERENCE_DIR = Path("reference_audio")

# Remove the stock demo voices so your voice is the only (and therefore default) option
for d in (VOICES_DIR, REFERENCE_DIR):
    d.mkdir(parents=True, exist_ok=True)
    for old_file in d.glob("*.wav"):
        old_file.unlink()

# Copy your Drive voice sample into both the predefined-voice folder and the
# reference/clone folder, so it works no matter which voice mode the UI is in.
shutil.copy(voice_path_in_drive, VOICES_DIR / VOICE_FILENAME)
shutil.copy(voice_path_in_drive, REFERENCE_DIR / VOICE_FILENAME)
print(f"  Installed {VOICE_FILENAME} as the only available voice")

CONFIG_PATH = Path("config.yaml")
with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

cfg.setdefault("model", {})
cfg["model"]["repo_id"] = "chatterbox"  # Default active model: Chatterbox Original (English)

cfg.setdefault("tts_engine", {})
cfg["tts_engine"]["default_voice_id"] = VOICE_FILENAME

cfg.setdefault("paths", {})
cfg["paths"]["output"] = str(DRIVE_OUTPUTS_DIR)

cfg.setdefault("audio_output", {})
cfg["audio_output"]["save_to_disk"] = True  # actually write generated wavs to the output folder above

cfg.setdefault("generation_defaults", {})
cfg["generation_defaults"].update(GENERATION_DEFAULTS)

cfg.setdefault("ui_state", {})
cfg["ui_state"]["last_text"] = "Type your text here."  # non-empty so the UI doesn't auto-load a preset and overwrite the sliders below
cfg["ui_state"]["last_voice_mode"] = "predefined"
cfg["ui_state"]["last_predefined_voice"] = VOICE_FILENAME
cfg["ui_state"]["last_reference_file"] = VOICE_FILENAME
cfg["ui_state"]["last_seed"] = GENERATION_DEFAULTS["seed"]
# [patched: chunk_size actually lives under ui_state, not a top-level "chunking"
# key — that key doesn't exist in the app's schema and was silently ignored.
# ui_state.last_chunk_size is what the web UI's chunk-size slider reads on
# load, and the slider's value is what gets sent as chunk_size on every
# /tts request (falls back to a default of 240 if this isn't set).
cfg["ui_state"]["last_chunk_size"] = CHUNK_SIZE
cfg["ui_state"]["last_split_text_enabled"] = True

with open(CONFIG_PATH, "w", encoding="utf-8") as f:
    yaml.safe_dump(cfg, f, default_flow_style=False, sort_keys=False)

print(f"  ✅ Active model set to: {cfg['model']['repo_id']} (Chatterbox Original / English)")
print(f"  ✅ Output directory set to: {DRIVE_OUTPUTS_DIR}")
print(f"  ✅ Default voice set to: {VOICE_FILENAME}")
print(f"  ✅ Generation defaults set to: {GENERATION_DEFAULTS}")

print("\n=== Removing old stdout log ===")
Path(LOG_STDOUT).unlink(missing_ok=True)

print("\n=== Starting server with LIVE logs ===")
print("Log file:", LOG_STDOUT)
print("To stop the server, run Cell 5.\n")

env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"

# Put HF cache somewhere inspectable/persistent for this runtime
env["HF_HOME"] = "/content/hf_home"
env["TRANSFORMERS_CACHE"] = "/content/hf_home/transformers"
env["HF_HUB_CACHE"] = "/content/hf_home/hub"
Path(env["HF_HOME"]).mkdir(parents=True, exist_ok=True)

proc = subprocess.Popen(
    ["/content/bin/micromamba", "run", "-n", "cb311", "python", "-u", "server.py"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=env,
)

import re

_progress_state = {"dots": 0, "last_current": -1}

def _handle_progress_line(raw_line):
    """
    Returns True if `raw_line` is a Chatterbox/tqdm 'Sampling: NN%|...' progress
    update (and should NOT be printed as-is). Instead, prints a '.' for each
    percent of progress, wrapping to a new line every 25 dots (so at most
    4 lines of dots appear per 1000-sample generation).
    """
    m = re.search(r"Sampling:.*?(\d+)/(\d+)\s*\[", raw_line)
    if not m:
        return False

    current, total = int(m.group(1)), int(m.group(2))

    # A new bar has started (current count dropped back down) — close out
    # the previous line of dots if it wasn't already at a 25-dot boundary.
    if current < _progress_state["last_current"]:
        if _progress_state["dots"] % 25 != 0:
            print()
        _progress_state["dots"] = 0

    _progress_state["last_current"] = current
    target_dots = min(100, int(current / total * 100)) if total else 0

    while _progress_state["dots"] < target_dots:
        print(".", end="", flush=True)
        _progress_state["dots"] += 1
        if _progress_state["dots"] % 25 == 0:
            print()

    return True

with open(LOG_STDOUT, "w", encoding="utf-8", errors="replace") as f:
    shown_link = False
    while True:
        line = proc.stdout.readline()
        if line:
            if not _handle_progress_line(line):
                print(line, end="")
            f.write(line)  # full detail still goes to the log file
            f.flush()

        if (not shown_link) and port_open():
            shown_link = True
            print("\n" + "="*60)
            print("=== Server is ready! ===")
            print("="*60)
            from google.colab.output import eval_js
            proxy_url = eval_js(f'google.colab.kernel.proxyPort({PORT})')
            print(f"\n🌐 Open this URL in a new browser tab:\n\n   {proxy_url}\n")
            print(f"📚 API docs:  {proxy_url}docs\n")
            print("="*60 + "\n")
            # Verify model load status via server endpoint
            try:
                mi = requests.get(f"http://127.0.0.1:{PORT}/api/model-info", timeout=2).json()
                print("\n/api/model-info:", mi)
            except Exception as e:
                print("\n/api/model-info query failed:", repr(e))

        if proc.poll() is not None:
            print("\n=== Server process exited with code", proc.returncode, "===")
            break

In [ ]:
%%bash
PORT=8004

echo "PIDs listening on port $PORT:"
sudo lsof -t -i:$PORT || true

echo "Killing..."
sudo lsof -t -i:$PORT | xargs -r sudo kill -9

echo "Verify nothing is listening:"
sudo lsof -i:$PORT || true
